In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


In [2]:
file_2023 = "DATA TABLE-2023- SALE 50.xlsx"
file_2024 = "DATA TABLE-2024- SALE  51.xlsx"
file_2025 = "DATA TABLE-2025- SALE  46- up to November.xlsx"

df_2023 = pd.read_excel(file_2023, sheet_name="DATA")
df_2024 = pd.read_excel(file_2024, sheet_name="DATA")
df_2025 = pd.read_excel(file_2025, sheet_name="DATA")

df_all = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

print("df_all shape:", df_all.shape)
print("Columns:", list(df_all.columns)[:20], "...")


df_all shape: (1503888, 33)
Columns: ['YEAR', 'MONTH', 'BROKER', 'LOT NO', 'SALE NO', 'INV NO', 'MF', 'REF', 'MARK', 'SALE TYPE', 'GRADE', 'QTY', 'PRICE', 'PRO', 'TYPE', 'ELE', 'BUYER CODE', 'BUYER NAME', 'CHEST', 'CATEGORY'] ...


In [3]:
def _find_col(df, candidates):
    m = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in m:
            return m[key]
    return None

def pick_price_col_best(df, after_filter_df=None):
    base = after_filter_df if after_filter_df is not None else df

    # 1) price-like numeric columns
    price_like = [c for c in df.columns if "price" in c.lower()]
    price_like = [c for c in price_like if pd.api.types.is_numeric_dtype(df[c])]
    if price_like:
        return max(price_like, key=lambda c: base[c].notna().sum())

    # 2) common names
    for c in ["AVG_PRICE", "avg_price", "PRICE", "price", "Average", "AVERAGE"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c

    # 3) last resort: numeric col with variance (avoid year/sale)
    avoid = set(["year", "sale", "sale no", "sale_no", "week", "week_id"])
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c.lower().strip() not in avoid]
    if not num_cols:
        raise ValueError("No numeric price column found.")
    return max(num_cols, key=lambda c: df[c].var(skipna=True))


In [4]:
def build_weekly_series(df_all, elev, grade):
    d = df_all.copy()

    elev_col  = _find_col(d, ["ELE1"])
    grade_col = _find_col(d, ["GRADE"])
    year_col  = _find_col(d, ["YEAR"])
    sale_col  = _find_col(d, ["SALE NO", "SALE_NO", "SALE", "SALE NO.", "SALE_NO."])

    if elev_col is None or grade_col is None or year_col is None or sale_col is None:
        raise ValueError(f"Missing required columns. Found: ELE1={elev_col}, GRADE={grade_col}, YEAR={year_col}, SALE={sale_col}")

    # normalize
    d[elev_col]  = d[elev_col].astype(str).str.strip().str.lower()
    d[grade_col] = d[grade_col].astype(str).str.strip().str.upper()

    elev_key  = str(elev).strip().lower()
    grade_key = str(grade).strip().upper()

    # elevation matching: exact OR contains ("low grown")
    mask_e = (d[elev_col] == elev_key) | (d[elev_col].str.contains(elev_key, na=False))
    d = d[mask_e & (d[grade_col] == grade_key)].copy()

    if d.empty:
        return pd.Series(dtype=float)

    # choose price col after filter
    price_col = pick_price_col_best(df_all, after_filter_df=d)

    # numeric convert
    d[year_col] = pd.to_numeric(d[year_col], errors="coerce")
    d[sale_col] = pd.to_numeric(d[sale_col], errors="coerce")
    d[price_col] = pd.to_numeric(d[price_col], errors="coerce")
    d = d.dropna(subset=[year_col, sale_col, price_col]).copy()

    d[year_col] = d[year_col].astype(int)
    d[sale_col] = d[sale_col].astype(int)

    # group by weekly key (YEAR, SALE NO)
    wk = d.groupby([year_col, sale_col])[price_col].mean().sort_index()

    # map to sequential safe weekly dates
    base = pd.Timestamp("2019-01-06")
    ds = base + pd.to_timedelta(np.arange(len(wk)) * 7, unit="D")

    y = pd.Series(wk.values, index=ds, name="price")
    return y


In [5]:
def split_train_test(y, test_weeks=52, min_train=30, min_test=10):
    y = y.dropna()
    if len(y) < (min_train + min_test):
        return None, None

    # auto test size
    tw = min(test_weeks, max(min_test, len(y)//3))
    return y.iloc[:-tw], y.iloc[-tw:]


In [6]:
PKL_MAP = {
    ("Low","ARIMA"):   "low_elevation_arima_models.pkl",
    ("Low","SARIMAX"): "low_elevation_sarimax_models.pkl",
    ("Low","PROPHET"): "low_elevation_prophet_models.pkl",

    ("Medium","ARIMA"):   "mid_elevation_arima_models.pkl",
    ("Medium","SARIMAX"): "mid_elevation_sarimax_models.pkl",
    ("Medium","PROPHET"): "medium_elevation_prophet_models.pkl",

    ("High","ARIMA"):   "high_elevation_arima_models.pkl",
    ("High","SARIMAX"): "high_elevation_sarimax_models.pkl",
    ("High","PROPHET"): "high_elevation_prophet_models.pkl",
}

CACHE = {}

def load_pkl(fname):
    with open(fname, "rb") as f:
        return pickle.load(f)

def get_models_dict(elev, modelname):
    key = (elev, modelname)
    if key not in CACHE:
        fname = PKL_MAP[key]
        if not os.path.exists(fname):
            raise FileNotFoundError(f"Missing file: {fname}")
        CACHE[key] = load_pkl(fname)
    return CACHE[key]

def extract_model_obj(entry):
    # if your dict value stores model inside another dict
    if isinstance(entry, dict):
        for k in ["model","fit","fitted","fitted_model","result","results","res"]:
            if k in entry:
                return entry[k]
    return entry


In [7]:
import numpy as np
import pandas as pd

def forecast_any(model_obj, y_train, y_test):
    """
    Works reliably for saved statsmodels models (ARIMA/SARIMAX) even if they were fit without datetime index.
    - Uses integer start/end positions for get_prediction
    - Falls back to forecast
    - Prophet uses ds dates
    """
    n = len(y_test)

    # ---------------------------
    # statsmodels ARIMA / SARIMAX
    # ---------------------------
    if hasattr(model_obj, "get_prediction"):
        try:
            start = len(y_train)
            end   = len(y_train) + n - 1
            pred = model_obj.get_prediction(start=start, end=end, dynamic=False).predicted_mean
            pred = np.asarray(pred, dtype=float)
            if len(pred) == n:
                return pred
        except Exception:
            pass

    if hasattr(model_obj, "forecast"):
        try:
            pred = model_obj.forecast(steps=n)
            return np.asarray(pred, dtype=float)
        except Exception:
            pass

    # ---------------------------
    # Prophet
    # ---------------------------
    if hasattr(model_obj, "predict"):
        df_future = pd.DataFrame({"ds": pd.to_datetime(y_test.index)})
        fcst = model_obj.predict(df_future)
        if "yhat" in fcst.columns:
            pred = fcst["yhat"].to_numpy(dtype=float)
        else:
            num_cols = fcst.select_dtypes(include=[np.number]).columns.tolist()
            pred = fcst[num_cols[0]].to_numpy(dtype=float)

        if len(pred) != n:
            pred = pred[:n] if len(pred) > n else np.pad(pred, (0, n-len(pred)), mode="edge")
        return pred

    raise ValueError("Unknown model type")


In [8]:
def metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    diff = y_true - y_pred
    mae  = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))
    mape = float(np.mean(np.abs(diff / np.clip(y_true, 1e-6, None))) * 100)
    return mae, rmse, mape


In [9]:
elev_dd = widgets.Dropdown(options=["Low","Medium","High"], value="Low", description="Elevation:")
grade_dd = widgets.Dropdown(options=[], description="Grade:")
btn_load = widgets.Button(description="Load Grades", button_style="info")
btn_run  = widgets.Button(description="Validate + Plot", button_style="success")
out = widgets.Output()

def load_grades(_=None):
    with out:
        clear_output(wait=True)
        elev = elev_dd.value

        # get grades from ARIMA dict (usually smaller)
        d = get_models_dict(elev, "ARIMA")
        grades = sorted(list(d.keys()))
        grade_dd.options = grades
        if grades:
            grade_dd.value = grades[0]

        print(f"✅ Grades loaded: {len(grades)}")

def run_validate(_=None):
    with out:
        clear_output(wait=True)

        elev = elev_dd.value
        grade = grade_dd.value

        # build REAL weekly series
        y = build_weekly_series(df_all, elev, grade)

        if y.empty:
            print("❌ No data after filtering. Check ELE1/GRADE labels.")
            return

        y_train, y_test = split_train_test(y, test_weeks=52)
        if y_train is None:
            print(f"❌ Not enough weekly points for {elev}-{grade}.")
            print("Total weekly points:", len(y))
            return

        rows = []
        preds = {}

        for modelname in ["ARIMA","SARIMAX","PROPHET"]:
            d = get_models_dict(elev, modelname)

            if grade not in d:
                print(f"⚠️ {modelname}: grade not found in saved dict")
                continue

            obj = extract_model_obj(d[grade])

            try:
                yhat = forecast_any(obj, y_train, y_test)
                yhat = pd.Series(yhat, index=y_test.index)

                mae, rmse, mape = metrics(y_test.values, yhat.values)
                rows.append([modelname, mae, rmse, mape])
                preds[modelname] = yhat

            except Exception as e:
                print(f"❌ {modelname} forecast failed:", e)

        if not rows:
            print("❌ No models could forecast.")
            return

        res = pd.DataFrame(rows, columns=["Model","MAE","RMSE","MAPE_%"]).sort_values("MAPE_%").reset_index(drop=True)
        best = res.iloc[0]["Model"]

        print(f"✅ BEST (lowest MAPE): {best}   |   {elev} — {grade}")
        print(f"Weekly points: total={len(y)}, train={len(y_train)}, test={len(y_test)}")
        display(res)

        # Plot
        plt.figure(figsize=(12,4))
        plt.plot(y_train.index, y_train.values, color="lightgray", label="Train (history)")
        plt.plot(y_test.index, y_test.values, linewidth=2, label="Test (actual)")

        for m in ["ARIMA","SARIMAX","PROPHET"]:
            if m in preds:
                plt.plot(preds[m].index, preds[m].values, linewidth=2 if m==best else 1, label=f"Pred: {m}")

        plt.title(f"{elev} elevation — {grade}: Real Auction Test vs Forecast")
        plt.xlabel("Week")
        plt.ylabel("Price")
        plt.legend()
        plt.tight_layout()
        plt.show()

btn_load.on_click(load_grades)
btn_run.on_click(run_validate)

display(widgets.VBox([widgets.HBox([elev_dd, btn_load, grade_dd, btn_run]), out]))
load_grades()


In [10]:
print("ELE1 unique (sample):", sorted(df_all[_find_col(df_all, ['ELE1'])].astype(str).str.lower().unique())[:20])
print("GRADE unique (sample):", sorted(df_all[_find_col(df_all, ['GRADE'])].astype(str).str.upper().unique())[:30])

# check BM low directly
y = build_weekly_series(df_all, "Low", "BM")
print("Low-BM weekly points:", len(y))
print(y.head())
print(y.tail())


ELE1 unique (sample): ['high', 'low', 'medium']
GRADE unique (sample): ['ABBFOSP', 'ABBOTSFORD', 'BABY TEA', 'BG01', 'BLACK FF', 'BLOOMING', 'BM', 'BOP', 'BOP1', 'BOP1A', 'BOPA', 'BOPF', 'BOPFSP', 'BOPSP', 'BP', 'BP1', 'BPS', 'BT', 'BT BOP', 'BT BOP1', 'BT BOPF', 'BT FBOP', 'BT OP', 'BT OP1', 'BTFINED1', 'BUD TEA', 'C.S FGS', 'C.S OP', 'C.S.FGS', 'C.S.OP']
Low-BM weekly points: 147
2019-01-06    850.513761
2019-01-13    868.943925
2019-01-20    860.932039
2019-01-27    863.146965
2019-02-03    870.560345
Name: price, dtype: float64
2021-09-26    805.894040
2021-10-03    804.293478
2021-10-10    795.503731
2021-10-17    817.157464
2021-10-24    809.326733
Name: price, dtype: float64
